# 自己回帰型ニューラルネットワークと自然言語処理


自己回帰型ニューラルネットワーク（Recurrent Neural Network：RNN）を用いた自然言語処理の記述方法を学習します．  
今回は長短期記憶（Long Short Tem Memory：LSTM）を持ったRNNを使います．

**目標：RNNを用いた自然言語処理の記述を理解**

---

例題では，PyTorchのWord2Vecを使って，回帰問題（文章生成）を解きます．  
演習では，GensimのWord2Vecを使って，分類問題（文章分類）を解きます．  
※[データのフォーマットが異なる](https://drive.google.com/file/d/1hmmDBO_-PFmtuB9XMBUXvEgrPb5_mS5D/view?usp=sharing)ので気をつけましょう．




---
## この教材について

「3分で学ぶPyTorch」シリーズの **NLP（自然言語処理） 基礎編（第2回）** の演習パートです。

このノートブックは**演習ファイル（task）**です。`【TASK】` と書かれた箇所を埋めて実行してください。詰まったときは同じ回の回答ファイル（ans）を参照してください。

この教材と関連記事は note で無料公開しています。
シリーズ一覧: https://note.com/technosend/m/m84d841b6d067

---

## 今回使う文章データ
これまでとは異なり，ごく短いデータを用意した．  

- 例題：文章生成での使い方
  - 1単語ずつ入力して文章ごとに，その文章の流れを学習させる
  - 学習データ：`新鮮な魚を焼く`, `珈琲牛乳が美味しい`　の2文  
  - テストデータ：なし
- 演習：文章分類での使い方
  - 1単語ずつ入力して文章ごとに，その文章が肯定的か否定的かを学習させる
    - 肯定的な文をクラス0，否定的な文をクラス1とする


|     分類    |  入力データ  |  教師データ  |
|     ---      |     ----      |      ----    |
|  学習データ  |  `今は焼肉を食べたい`  |  0  |
|             |  `魚は一生食べたくない`  |  1  |
| テストデータ  |  `珈琲牛乳が好きだ`  |  0  |
|             |  `辛いものは苦手だ`  |  1  |

## 前準備

形態素解析を扱うためのライブラリ（MeCab）とWord2Vecを扱うためのライブラリ（Gensim）をColabratoryにインストールし，Word2Vecの学習済みモデルのダウンロード，データセットの前処理で使う関数群を用意する．

---


### 前準備1. MeCabとGensimのインストール

形態素解析を扱うためのライブラリ（MeCab）とWord2Vecを扱うことができるライブラリ（Gensim）をインストール，インポートする．

---


#### 前準備1のコマンド

In [ ]:
!pip install gensim -q
!pip install mecab-python3 -q  # バージョン固定解除 (0.996.6rc2は古い)
# mecab-python3 1.0以降はシステムMeCabのインストール不要
from gensim.models.word2vec import Word2Vec
import MeCab
from IPython.display import clear_output

### 前準備2. Word2Vecの学習済みモデルのダウンロードと読み込み

Word2Vecの学習済みモデルである白ヤギモデルをダウンロードし読み込む．

#### 前準備2のコード

In [ ]:
# 白ヤギモデルのダウンロードと解凍
# 下記URLは白ヤギコーポレーション(2017年公開)のS3バケットです。
# アクセス不可の場合は代替モデル(chiVe等)の使用を検討してください。
# 参考: https://github.com/WorksApplications/chiVe
!wget "http://public.shiroyagi.s3.amazonaws.com/latest-ja-word2vec-gensim-model.zip"
!unzip "latest-ja-word2vec-gensim-model.zip"
clear_output()

# シロヤギモデルの読み込み
model = Word2Vec.load("word2vec.gensim.model")
wv = model.wv

# ベクトルの次元数を確認
embedding_size = wv.vector_size
print("ベクトルの次元数 :", embedding_size)

  - 白ヤギモデルをダウンロードする
    - ダウンロード元：白ヤギコーポレーションのブログ[word2vecの学習済み日本語モデルを公開します](https://aial.shiroyagi.co.jp/2017/02/japanese-word2vec-model-builder/) より

  - ダウンロードした白ヤギモデルを読み込み，今回の例題・演習で使用するパラメータを取得する  
    - wv：白ヤギモデルを読み込んだWord2Vec
    - embedding_size：単語をベクトルへ変換したときのベクトルの次元数

### 前準備3. functionsのダウンロードとインポート

文章の前処理で用いる関数が格納されたモジュールfunctionsをダウンロードし，インポートする．  

※functionsはこの講義専用に作成したので，PyPiなどのパッケージ管理ツールで公開されているものではないので注意．

---


#### 前準備3のコード

In [ ]:
!pip install gdown --upgrade -q  # gdown 4.6以降はアップグレード推奨
import gdown
# 旧形式URL変換済み: uc?export=download は gdown 4.6以降で不安定
file_id = "182oAY1TFgcer5MLAqk5tBK0e7HoCI7Um"  # Google Drive ファイルID
gdown.download(id=file_id, output="functions.py", quiet=True)
clear_output()

import functions

print("ベクトル化のプロセスを試してみる")
# 1. データの用意
text_list = ["ここは夜景が綺麗です", "朝日が眩しい"]

# 2. 各文章を分かち書き
text_list = functions.word_tokenize(text_list)
print("分かち書きした結果", text_list)

# 3. 単語列をWord2Vecで変換
text_list = functions.text_to_word2vec(text_list, wv)
print("Word2Vecでベクトル化した結果", text_list[0][0])

##### functionsを使った前処理の方法

前処理としては，文章の分かち書きやWord2Vecによるベクトル化を行う．



  1. データの用意
    - 処理する文章をlist形式で宣言  
    ```python
    text_list = ["ここは夜景が綺麗です", "朝日が眩しい"]
    ```

  2. 各文章を分かち書き
    - 各文章に対して分かち書きを行い，単語間がスペースで区切られた文章（単語列）を作成
    ```python
    text_list = functions.word_tokenize(text_list)
    # 第１引数：分かち書きを行う文章(list)
    # 戻り値1：分かち書きした結果(list)
    ```

  3. 単語列をWord2Vecでベクトル化
    - 単語列の各単語をWord2Vecでそれぞれベクトル化
    ```python
    text_list = functions.text_to_word2vec(text_list, wv)
    # 第1引数：ベクトル化する単語列(list)
    # 第2引数：変換に使うWord2Vecのモデル(Word2VecKeyedVectors)
    # 戻り値1：Word2vecでベクトル化した文章(list)
    ```


## 演習 文章分類

**LSTM1層・全結合1層のRNNの作成**  
文章を単語に分解し時系列データとして順次入力し，文章の分類を行う分類問題を解く．



### 演習1. ライブラリのインポート


深層学習演算ライブラリPyTorchなどのライブラリ，パッケージ，モジュールをインポートする．  

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=14XdT7XWs6JzTil6JhZZw7aM3VpAxdYH2&sz=w400">


#### 演習1のコード

In [ ]:
# 演習1. ライブラリのインポート

import torch
import torch.nn as nn
import torch.optim as optim

#### 今回使うパッケージ，ライブラリ，モジュール一覧

- torch：多次元テンソルのデータ構造とそのテンソルのための算術演算が組み込まれたパッケージ
- torch.nn：ニューラルネットワークを宣言するためのパッケージ  
nnという略称を与えることが多い
- torch.optim：最適化器を宣言するためのパッケージ  
optimという略称を与えることが多い

<font color="blue">【TASK】</font>パッケージをインポートしましょう

前処理でインポート済みのもの
- **<font color="red">【NEW!】</font>**MeCab：形態素解析を行うライブラリ（白ヤギモデルに合わせたバージョンを指定）
- **<font color="red">【NEW!】</font>**gensim：Word2Vecを扱うためのライブラリ
- functions：前準備3でダウンロード，インポートしたもの  
  文章を分かち書きしたりWord2Vecに変換したりする関数が格納されたモジュール  
  ※この講義専用のモジュールで，PyPiなどのパッケージ管理ツールで公開されているものではないので注意


### 演習2. ニューラルネットワークの宣言

ニューラルネットワーククラスを宣言して，そのクラスのインスタンスを宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1jl1W_RW7HrU0ksk1a0XrSq6CyldXF4qZ&sz=w400">


#### 演習2のコード

In [ ]:
# 演習2. ネットワークの宣言

# 1.ニューラルネットワーククラスの宣言
class TextClassifier(nn.Module):
    def __init__(self, embedding_size):
        super(TextClassifier, self).__init__()
        # 【TASK】LSTM層と全結合層を宣言

    def forward(self, x):
        # 【TASK】順伝播のパスを宣言
        return x
    
# 2.インスタンスの宣言
text_classifier = #【TASK】ニューラルネットワーククラスのインスタンスの宣言
device = #【TASK】GPUの指定
# 【TASK】GPUにセットアップ

#### 1. ニューラルネットワーククラスの宣言  
  - nn.Moduleを継承したクラス「TextClassifier」を宣言
    - コンストラクタの引数にembedding_sizeを持つ
        - embedding_size：単語からベクトルへ変換したときのベクトルの次元数
  - \_\_init\_\_()とforward()を宣言

  ```python
  # 1.ニューラルネットワーククラスの宣言
  class TextClassifier(nn.Module):
      def __init__(self, embedding_size):
          super(TextClassifier, self).__init__()
          # 【TASK】LSTM層と全結合層を宣言
      def forward(self, x):
          # 【TASK】順伝播のパスを宣言
          return x
  ```


<font color="blue">【TASK】</font>LSTM層と全結合層をnn.LSTMクラスとnn.Linearクラスを用いて宣言しましょう  
ニューラルネットワークの構成は下記の通りです
- LSTM層１：入力embedding_size，出力embedding_size，レイヤー数1
- 全結合層1：入力embedding_size，出力2  

<font color="blue">【TASK】</font>順伝播のパスを宣言しましょう  
ある時刻のデータ（数値）を入力すると想定し，順伝播のパスの構成は下記の通りです  
1. LSTM層1
2. 全結合層1  

※PyTorchのLSTM層は内部に活性化関数を持つので，今回LSTM層と全結合層の間に活性化関数がありません．


#### 2. インスタンスの宣言

- text_classifierという名前でインスタンスを宣言
    ```python
    text_classifier = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
    ```

- GPUにセットアップ

    ```python
    device = # 【TASK】GPUの指定
    # 【TASK】GPUにセットアップ
    ```


  - 今回はWord2Vecのモデルや学習する文章によってニューラルネットワークの構成が変わるので，引数でパラメータを渡す
  ```python
  text_classifier = TextClassifier(embedding_size)
  # 第1引数：学習済みWord2Vecモデルが出力するベクトルの次元数(int)
  ```

- ニューラルネットワーククラスのインスタンスの宣言

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスを宣言しましょう 



- GPUの指定
  - デバイス指定用の変数deviceを宣言しましょう
  - [torch.device](https://pytorch.org/docs/stable/tensor_attributes.html#torch.torch.device)クラスを使って変数deviceを初期化しましょう
  - torch.deviceクラス宣言時に"cuda:0"を与えましょう

  ```python
  device = torch.device("cuda:0")
  ```

　　<font color="blue">【TASK】</font>GPUを指定しましょう

- GPUにセットアップ
  - Moduleクラスにもto()がある
  - Tensor型変数同様にto()を使ってGPUにデータを渡すことができる
  - 学習時にGPUを使う場合は，ニューラルネットワーククラスのインスタンスをGPUに渡す

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスをGPUにセットアップしましょう  
  - [to()](https://pytorch.org/docs/1.9.1/generated/torch.Tensor.to.html)を使ってGPUにセットアップしましょう
  ```python
  text_classifier.to(device)
  ```


### 演習3. 誤差関数・最適化器の設定

誤差関数と最適化器を宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1oS8_oitSvcQ9f5_uYMrDXyC3P_vrM7jp&sz=w400">


#### 演習3のコード

In [ ]:
# 演習3. 誤差関数・最適化器の設定

# 1.誤差関数の宣言
criterion = # 【TASK】誤差関数の宣言

# 2.最適化器の宣言
optimizer_text_classifier = # 【TASK】最適化器の宣言

#### 1. 誤差関数の宣言

- クロスエントロピー誤差を計算するcriterionを宣言

    ```python
    criterion = # 【TASK】誤差関数の宣言
    ```


<font color="blue">【TASK】</font>誤差関数を宣言しましょう  
- クロスエントロピー誤差関数を使いましょう
- nn.CrossEntropyLossクラスを用いて宣言しましょう

#### 2. 最適化器の宣言

- Adamを計算するoptimizer_text_classifierを宣言

    ```python
    optimizer_text_classifier = # 【TASK】最適化器の宣言
    ```

<font color="blue">【TASK】</font>最適化器を宣言しましょう  
- Adamを使いましょう
- [optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#adam)クラスを用いて宣言しましょう  
- 引数の構成は下記の通りです  
  - ニューラルネットワークのパラメータtext_classifierのパラメータ

### 演習4. データセットの準備

自作した関数make_dataset()を使ってデータセットを作成する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=17fS-oMI83rSL7SxN_GyKHjxP-FO-R3aQ&sz=w400">


#### 演習4のコード

In [ ]:
# 演習4. データセットの作成

# 1. データセットを作成する関数の定義
def make_dataset(word2vec_list, text_class, train_num=2):
    # 入力データをTensor型に変換
    input = torch.tensor(word2vec_list)
    # バッチ，データ長，データの順番になるように並べ替え
    input = input.permute(1,0,2)
    
    # 教師データをTensor型に変換
    label = torch.tensor(text_class, dtype=torch.int64)
    
    # 入力データと教師データをひとつにまとめる
    train_loader = [(input[:, :train_num], label[:train_num])]
    test_loader = [(input[:, train_num:], label[train_num:])]

    # データの確認
    print("入力データの次元数 :", input[:, :train_num].shape)
    print("入力データの最後データの中身（Word2Vec） :", input[:, :train_num][-1])
    print("教師データの次元数 :", label[:train_num].shape)
    print("教師データの中身（クラスのラベル） :", label[:train_num])

    return train_loader, test_loader

# 2. 文章とクラスの用意
text_list = ["今日は焼肉を食べたい","魚は一生食べたくない", "珈琲牛乳が好きだ", "辛いものは苦手だ"]
text_class = [0, 1, 0, 1]

# 3. 各文章を分かち書き
text_list = functions.word_tokenize(text_list)

# 4. 分かち書きされた文章をWord2Vecで変換
text_list = functions.text_to_word2vec(text_list, wv)

# 5. データセットの作成
train_loader, test_loader = make_dataset(text_list, text_class)

#### 1. データセットを作成する関数の定義
  - make_dataset()という名前でデータセットを作成する関数を定義  
    - 入力データ用の変数，教師データ用の変数，学習データの数を受け取る
    - 学習データのデータローダーとテストデータのデータローダーを返す  

    ```python
    train_loader, test_loader = make_dataset(word2vec_list, text_class, train_num=2) 
    # 第1引数：入力データ用の変数，　ベクトル化した文章(list)
    # 第2引数：教師データ用の変数，　入力データに対応したクラスのラベル(list)
    # 第3引数：学習データの数(int), デフォルト2
    # 戻り値1：学習データのデータローダー
    # 戻り値2：テストデータのデータローダー
    ```
  

  - 入力データとして，ベクトル化した文章をTensor型に変換
    - Tensor型に変換
    ```python
    input = torch.tensor(word2vec_list)
    ```
    - バッチ，データ長，データの順番になるように並べ替え
    ```python
    input = input.permute(1,0,2)
    ```
  - 教師データをTensor型に変換
    ```python
    label = torch.tensor(text_class, dtype=torch.int64)
    ```
  - 入力データと教師データをひとつにまとめる
    - 学習データとして入力データと教師データをtrain_num個（ここでは2個）取り，残りをテストデータとする
    - それぞれ入力データと教師データをtuple形式でまとめ，さらにlist形式でまとめ直す
    ```python
    train_loader = [(input[:, :train_num], label[:train_num])]
    test_loader = [(input[:, train_num:], label[train_num:])]
    ```


#### 2. 文章とクラスの用意

  - text_listという名前で文章をlist形式で宣言
  - text_classという名前で対応するクラスのラベルをlist形式で宣言

  ```python
  text_list = # 文章を宣言
  text_class = # 文章のクラスのラベルを宣言
  ```

  - 用意する文章は`今は焼肉を食べたい`，`魚は一生食べたくない`，`珈琲牛乳が好きだ`，`辛いものは苦手だ`とする
  - クラスは，肯定的な文章をクラス0とし，否定的な文章をクラス1とする
    - `今は焼肉を食べたい`：クラス0
    - `魚は一生食べたくない`：クラス1
    - `珈琲牛乳が好きだ`：クラス0
    - `辛いものは苦手だ`：クラス1
 

#### 3. 各文章を分かち書き
  
  - text_listという名前で文章を分かち書きした単語列を格納する変数を宣言

  ```python
  text_list = # 文章を分かち書き
  ```


  - 分かち書きにはfunctions.word_tokenize()を使う
  ```python
  text_list = functions.word_tokenize(text_list)
  # 第1引数：分かち書きする文章(list)
  # 戻り値1：分かち書きした結果の文章データ(list)
  ```

#### 4. 分かち書きされた文章をWord2Vecでベクトル化
  - text_listにWord2Vecでベクトル化したlist形式の配列を再代入

  - Word2Vecによるベクトル化にはfunctions.text_to_word2vec()を使う
  ```python
  text_list = functions.text_to_word2vec(text_list, wv)
  # 第1引数：ベクトル化する単語列(list)
  # 第2引数：変換に使うWord2Vecのモデル(Word2VecKeyedVectors)
  # 戻り値1：Word2vecでベクトル化した文章(list)
  ```


#### 5. データセットの作成

  - train_loaderという名前で学習データのデータローダー用の変数を，  
  test_loaderという名前でテストデータのデータローダー用の変数を宣言  
  - make_dataset()を用いてデータセットを作成
  
  ```python
  train_loader, test_loader = # データセットの作成
  ```


### 演習5. 学習

教師データとの誤差を計算し，パラメータを更新する．  

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1A6TjumoBejWpKGvD6TDQeivN1tpeEBp4&sz=w400">


#### 演習5のコード：前半の学習部分

In [ ]:
# 演習5. 学習

# 1. 学習ループの作成
epochs = # 【TASK】エポック数

# エポックのループ
for epoch in # 【TASK】エポック:
    # 学習データのデータローダーのループ
    for data in # 【TASK】学習データのデータローダー:

        # 2. ニューラルネットワークへのデータの入力
        # 文章とラベルに分割
        inputs, labels = data
        inputs = # 【TASK】学習データをGPUにセットアップ
        labels = # 【TASK】教師データをGPUにセットアップ
        outputs = # 【TASK】ニューラルネットワークからの出力

        # 3. 誤差の計算（BPTTパターン2：最後の時刻に教師データがある場合）
        loss = # 【TASK】誤差計算

        # 4. 誤差逆伝播とパラメータの更新
        # 【TASK】パラメータの微分値を初期化
        # 【TASK】誤差逆伝播
        # 【TASK】パラメータの更新

        # 現在の誤差の値の表示
        if (epoch + 1) % 100 == 0:
            print("epoch :", epoch + 1, "学習誤差 :",loss.item())

#### 1. 学習ループの作成  
- ミニバッチ学習を行う学習ループの作成
- epochsという名前でエポック用の変数を宣言
- 外側にエポック，内側に学習データのデータローダーのループを作成
```python
epochs = # 【TASK】エポック数
for epoch in # 【TASK】エポック
      for data in # 【TASK】学習データのデータローダー
```

<font color="blue">【TASK】</font>学習ループを作成しましょう  
ループの設定は下記の通りです
- エポック
  - エポック数1000
      - rangeクラスを使ってループさせましょう
  - 学習データのデータローダー
    - ループ対象：学習データのデータローダーtrain_loader
    - forを使ってtrain_loaderをループさせましょう

#### 2. ニューラルネットワークへのデータの入力  

- GPUにセットアップ
- outputsという名前で出力用の変数を宣言
```python
# 文章とラベルに分割
inputs, labels = data
inputs = # 【TASK】学習データをGPUにセットアップ
targets = # 【TASK】教師データをGPUにセットアップ
outputs = # 【TASK】ニューラルネットワークからの出力
```


  
<font color="blue">【TASK】</font>ニューラルネットワークへデータを入力しましょう
  - 入力inputsをGPUにセットアップしましょう
  - 教師データlabelsをGPUにセットアップしましょう
  - inputsをニューラルネットワークに与えて出力outputsを取得しましょう

#### 3. 誤差の計算（BPTTパターン2：最後の時刻に教師データがある場合）

  - lossという名前で誤差計算の結果用の変数を宣言
  ```python
  loss = # 【TASK】誤差計算
  ```



- 誤差計算はnn.MSELossクラスやnn.CrossEntropyLossクラスなどの誤差関数クラスのインスタンスに引数を2つ与えて行う
```python
loss = criterion(outputs, labels)
# 第1引数：ニューラルネットワークの出力
# 第2引数：教師データ
```

<font color="blue">【TASK】</font>誤差を計算(最後の時刻に教師データがある場合）をしましょう  
設定は以下の通りです．
- 出力，教師データともに最後の時刻（シーケンスの最後）のデータ抜き出す
- 配列の最後のデータは-1を指定して抜き出す

#### 4. 誤差逆伝播とパラメータの更新  

- パラメータの微分値を初期化
- 誤差逆伝播
- パラメータの更新
```python
# 【TASK】パラメータの微分値を初期化
# 【TASK】誤差逆伝播
# 【TASK】パラメータの更新
```


<font color="blue">　【TASK】</font>誤差逆伝播とパラメータの更新を行いましょう
- zero_grad()を使ってパラメータの微分値の初期化をしましょう  
  - zero_grad()は最適化器が持っているので次のようにして呼び出します
  ```python
  optimizer_text_classifier.zero_grad()
  ```

- backward()を使って誤差逆伝播させましょう
  - backward()は計算結果を持った変数から次のようにして呼び出します
  ```python
  loss.backward()
  ```

- step()を使ってパラメータの更新を行いましょう
  - パラメータの更新は最適化器の持つstep()は最適化器が持っているので次のようにして呼び出します
  ```python
  optimizer_text_classifier.step()
  ```





#### 演習5のコード：後半の予測部分

In [ ]:
# 5. 文章の分類

for data in test_loader:
  
    # ニューラルネットワークへのデータの入力
    inputs, labels = data
    inputs = inputs.to(device)
    print("inputs shape", inputs.shape)
    outputs = text_classifier(inputs)
    print("outputs shape", outputs.shape)

    # 予測ラベルを取得
    pred = outputs.argmax(axis=2)
    print("pred shape", pred.shape)

    # 出力の表示
    print("予測ラベル :", pred[-1][0].item(), ", 正解ラベル :", labels[0].item())
    print("予測ラベル :", pred[-1][1].item(), ", 正解ラベル :", labels[1].item())

#### 5. 文章の分類

- 単語区切りの文章を順次ニューラルネットワークへ入力し，文章単位で分類する

- ニューラルネットワークへのデータの入力
  - ニューラルネットワークへデータを入力し出力を得る
  - 入力がデータ⻑ × バッチ × データになっているのを確認する
  - 出力がデータ⻑ × バッチ × 予測結果になっているのを確認する

- 予測ラベルを取得
  - 予測は確率の大きい方を取得する
  - 最大値のインデックスの取得にはargmax()を使う
  - データ⻑ × バッチ × 予測結果なので予測結果に該当する部分（axis=2）を指定する
- 出力の表示
  - 時系列の最後のデータがその時系列の分類結果を取得する
  - データ⻑ × バッチなので，時系列の最後は[-1]で指定
  - それぞれのバッチごとに表示

<font color="blue">【TASK】</font>文章を分類してみましょう．

## まとめ

今回は，例題では次の単語を予測することで文章生成し，演習では文章分類しました．  
理解のために簡単な短文章を学習させましたが，次の段階として長文章や文章数を増やすことが挙げられます．  
併せてエポック数やNNの構造を変えたりすることで学習精度を向上させることができます．  
余力があればこれらの取り組みも行ってみてください．